# Meta Thinker: Game24 Deep Dive

This notebook focuses specifically on the **24-Point Game**, a challenging arithmetic puzzle that demonstrates the power of adaptive reasoning strategies.

## The 24-Point Game

**Objective**: Use four (or more) given numbers with basic arithmetic operations (+, -, ×, ÷) to create an expression that equals exactly 24.

**Rules**:
- Use each number exactly once
- Use only basic arithmetic operations
- Parentheses are allowed
- The final result must equal exactly 24

**Example**: Given numbers 1, 2, 3, 4
- Solution: (1 + 2 + 3) × 4 = 24
- Or: 4 × 3 × (2 / 1) = 24

## Why Game24 is Interesting

The Game24 puzzle is ideal for testing reasoning strategies because:
1. **Multiple solutions exist** - encourages exploration
2. **Requires planning** - can't just try random operations
3. **Has dead ends** - some paths don't lead to 24
4. **Benefits from backtracking** - when a path fails, try a different approach
5. **Clear success criteria** - either you get 24 or you don't

## What We'll Demonstrate

1. Basic Game24 solving with different reasoning strategies
2. Meta-reasoning: letting the model choose its own strategy
3. Strategy invention: creating new reasoning approaches
4. Learning from examples: few-shot learning for Game24
5. Handling challenging cases: 5+ numbers, difficult combinations

## Setup and Imports

In [ ]:
from utils.utils import *
from utils.agent import *
import os
import json
import numpy as np

print("✓ Imports successful")

## Initialize LLM Client

In [ ]:
# Initialize GPT-4 client (you can also use 'deepseek')
client = initialize_pipeline('gpt')

print("✓ Client initialized")

## 1. Basic Game24 Example

Let's start with a simple example to see how the model approaches the problem.

In [ ]:
# Create a simple Game24 prompt
simple_prompt = [
    {
        'role': 'system',
        'content': 'You are a helpful assistant skilled in math reasoning and puzzle solving.'
    },
    {
        'role': 'user',
        'content': 'Play a 24-point game with me. Use the numbers 1, 2, 3, and 4.'
    }
]

# Get response
basic_response = pipeline_message(simple_prompt, client)

print("=" * 70)
print("BASIC GAME24 SOLUTION:")
print("=" * 70)
print(basic_response)
print("=" * 70)

## 2. Meta-Reasoning System Prompt

Now let's define a comprehensive system prompt that teaches the model to:
- Understand different reasoning strategies
- Track trials and learn from failures
- Invent new strategies when needed

In [ ]:
GAME24_META_PROMPT = """You are a strategic reasoning agent tasked with solving the 24-point arithmetic game.

Game Rules:
- You are given a list of numbers (usually 4 or 5)
- Your goal is to use all of the given numbers exactly once
- You may use the following operations: addition (+), subtraction (−), multiplication (×), and division (÷)
- You may use parentheses to control operation order
- Your final expression must evaluate exactly to 24
- You may not introduce any new numbers or reuse a number more than once

Available Reasoning Strategies:

1. **Chain of Thought (CoT)**:
   A simple step-by-step reasoning process. Progress linearly, breaking the problem into small steps.
   "Let's think step by step."

2. **Tree of Thought (ToT)**:
   A parallel, multi-path exploration. Imagine three experts independently solving the problem. 
   Each expert writes down one reasoning step, then they share and compare. After each round, 
   experts continue based on their branch. If an expert realizes their reasoning is flawed, they drop out.
   "Imagine three experts solving the problem in parallel, exchanging reasoning at each step."

3. **Algorithm of Thought (AoT)**:
   A structured, simulation-based process. First, perform forward analysis: reason step-by-step. 
   Then, use backtracking to verify the result, working from the goal backward.
   "First analyze, then backtrack to ensure correctness."

Your Objectives:
1. Choose the most appropriate reasoning style based on problem complexity
2. If existing styles fail or seem insufficient, invent a new reasoning style and explain its purpose
3. Record your past trials and reasoning attempts to avoid repeating failed paths
4. Implement meaningful search:
   - If a trial yields a near miss (e.g., 22 or −24), try small modifications instead of starting over
   - Avoid purely random or brute-force strategies

Response Format:
<strategy>
[Selected strategy: CoT, ToT, AoT, or a newly invented one]
</strategy>

<think>
[Explain your reasoning process. Reflect on failures and what you've learned.]
</think>

<trials>
[List all expressions you've tried and their results. Example: "(5 + 5) × 2 = 20"]
</trials>

<answer>
[Final valid expression that evaluates exactly to 24]
</answer>
"""

print("✓ Meta-reasoning prompt defined")
print(f"Prompt length: {len(GAME24_META_PROMPT)} characters")

## 3. Load Game24 Dataset

Load existing Game24 problems with annotated solutions to use for few-shot learning.

In [ ]:
# Load Game24 dataset
try:
    with open('./input_data/qa_dataset_game24.json', 'r') as file:
        game24_data = json.load(file)
    
    print(f"✓ Loaded Game24 dataset: {len(game24_data)} problems")
    
    # Display example
    if len(game24_data) > 0:
        example = game24_data[0]
        print("\nExample problem:")
        print(f"  Question: {example['question']}")
        print(f"  Best strategy: {example['best']}")
        print(f"  Solution preview: {example[example['best']][0][:100]}...")

except FileNotFoundError:
    print("⚠ Dataset file not found. Continuing with manual examples...")
    game24_data = []

## 4. Create Few-Shot Learning Prompt

Sample examples from the dataset to teach the model through demonstration.

In [ ]:
def create_game24_few_shot(data, n_examples=10):
    """
    Create a few-shot learning prompt for Game24.
    
    Args:
        data: List of Game24 problems with solutions
        n_examples: Number of examples to include
    
    Returns:
        List of message dictionaries
    """
    messages = []
    
    # Add system prompt
    messages.append({
        'role': 'system',
        'content': GAME24_META_PROMPT
    })
    
    if len(data) == 0:
        print("⚠ No dataset available for few-shot learning")
        return messages
    
    # Sample random examples
    n_samples = min(n_examples, len(data))
    indices = np.random.randint(0, len(data), n_samples)
    
    # Add introduction
    messages.append({
        'role': 'user',
        'content': (
            f"I will show you {n_samples} examples from the Game24 dataset. "
            "Each example shows the optimal reasoning strategy and solution."
        )
    })
    
    # Add examples
    for idx in indices:
        example = data[idx]
        question = example['question']
        best_strategy = example['best']
        solution = example[best_strategy][0]
        
        messages.append({
            'role': 'system',
            'content': f"Example: {question}"
        })
        
        messages.append({
            'role': 'user',
            'content': (
                f"Using the reasoning style '{best_strategy}', the answer is:\n\n{solution}"
            )
        })
    
    print(f"✓ Created few-shot prompt with {n_samples} examples")
    return messages


# Create few-shot messages
if len(game24_data) > 0:
    few_shot_messages = create_game24_few_shot(game24_data, n_examples=10)
else:
    few_shot_messages = [{'role': 'system', 'content': GAME24_META_PROMPT}]
    print("⚠ Using system prompt without examples")

## 5. Test Meta-Reasoning on Standard 4-Number Problems

Let's test the meta-reasoning approach on a standard 4-number Game24 problem.

In [ ]:
# Test problem: 1, 4, 6, 1
test_numbers = "1 4 6 1"

test_messages = few_shot_messages.copy()
test_messages.append({
    'role': 'user',
    'content': f"Solve this Game24 problem: {test_numbers}"
})

print(f"Testing with numbers: {test_numbers}\n")
response_standard = pipeline_message(test_messages, client)

print("=" * 70)
print("META-REASONING SOLUTION (Standard 4 numbers):")
print("=" * 70)
print(response_standard)
print("=" * 70)

## 6. Parse Response Components

Extract strategy, thinking process, trials, and final answer.

In [ ]:
def parse_game24_response(response: str) -> dict:
    """
    Parse the structured Game24 response.
    
    Args:
        response: Full response with XML-like tags
    
    Returns:
        Dictionary with parsed components
    """
    result = {
        'strategy': '',
        'thinking': '',
        'trials': '',
        'answer': ''
    }
    
    # Extract strategy
    if '<strategy>' in response and '</strategy>' in response:
        start = response.find('<strategy>') + len('<strategy>')
        end = response.find('</strategy>')
        result['strategy'] = response[start:end].strip()
    
    # Extract thinking
    if '<think>' in response and '</think>' in response:
        start = response.find('<think>') + len('<think>')
        end = response.find('</think>')
        result['thinking'] = response[start:end].strip()
    
    # Extract trials
    if '<trials>' in response and '</trials>' in response:
        start = response.find('<trials>') + len('<trials>')
        end = response.find('</trials>')
        result['trials'] = response[start:end].strip()
    
    # Extract answer
    if '<answer>' in response and '</answer>' in response:
        start = response.find('<answer>') + len('<answer>')
        end = response.find('</answer>')
        result['answer'] = response[start:end].strip()
    
    return result


# Parse the response
parsed = parse_game24_response(response_standard)

print("\n" + "=" * 70)
print("PARSED COMPONENTS:")
print("=" * 70)
print(f"\nStrategy Used: {parsed['strategy']}")
print(f"\nTrials Attempted:\n{parsed['trials'][:300]}..." if len(parsed['trials']) > 300 else f"\nTrials Attempted:\n{parsed['trials']}")
print(f"\nFinal Answer: {parsed['answer']}")
print("=" * 70)

## 7. Test Strategy Invention: Challenging 5-Number Problem

Now let's test with 5 numbers - a harder problem that may require inventing a new strategy.

In [ ]:
# Challenging 5-number problem
challenging_numbers = "5 5 5 1 1"

challenging_messages = few_shot_messages.copy()
challenging_messages.append({
    'role': 'user',
    'content': (
        f"Solve this harder Game24 problem with 5 numbers: {challenging_numbers}\n\n"
        "If the existing reasoning styles don't work well, feel free to invent a new one."
    )
})

print(f"Testing with 5 numbers: {challenging_numbers}\n")
response_challenging = pipeline_message(challenging_messages, client)

print("=" * 70)
print("META-REASONING SOLUTION (5 numbers):")
print("=" * 70)
print(response_challenging)
print("=" * 70)

In [ ]:
# Parse the challenging response
parsed_challenging = parse_game24_response(response_challenging)

print("\n" + "=" * 70)
print("ANALYSIS OF CHALLENGING PROBLEM:")
print("=" * 70)
print(f"\nStrategy: {parsed_challenging['strategy']}")

# Check if a new strategy was invented
known_strategies = ['CoT', 'ToT', 'AoT', 'Chain of Thought', 'Tree of Thought', 'Algorithm of Thought']
is_novel = not any(known in parsed_challenging['strategy'] for known in known_strategies)

if is_novel:
    print("\n✨ NEW STRATEGY INVENTED! ✨")
    print(f"Novel strategy: {parsed_challenging['strategy']}")
else:
    print("\n✓ Used existing strategy")

print(f"\nFinal Answer: {parsed_challenging['answer']}")
print("=" * 70)

## 8. Iterative Refinement Example

Show how the model can learn from incorrect attempts.

In [ ]:
# Create an iterative conversation
iterative_messages = few_shot_messages.copy()

# First attempt
iterative_messages.append({
    'role': 'user',
    'content': "Solve this Game24 problem: 3 3 8 8"
})

print("First attempt...\n")
response_1 = pipeline_message(iterative_messages, client)

print("=" * 70)
print("FIRST ATTEMPT:")
print("=" * 70)
print(response_1[:400] + "..." if len(response_1) > 400 else response_1)
print("=" * 70)

# Extract first answer
parsed_1 = parse_game24_response(response_1)
print(f"\nFirst Answer: {parsed_1['answer']}")

# If the answer seems wrong or can be improved, ask for refinement
iterative_messages.append({'role': 'assistant', 'content': response_1})
iterative_messages.append({
    'role': 'user',
    'content': (
        "Can you verify this solution is correct and all numbers are used exactly once? "
        "If there's an error, please correct it."
    )
})

print("\n\nVerification attempt...\n")
response_2 = pipeline_message(iterative_messages, client)

print("=" * 70)
print("VERIFICATION RESPONSE:")
print("=" * 70)
print(response_2[:400] + "..." if len(response_2) > 400 else response_2)
print("=" * 70)

## 9. Batch Testing on Multiple Problems

Test the system on multiple Game24 problems systematically.

In [ ]:
def batch_test_game24(client, problems: list, few_shot_messages: list):
    """
    Test multiple Game24 problems.
    
    Args:
        client: LLM client
        problems: List of number combinations (as strings)
        few_shot_messages: Few-shot learning messages
    
    Returns:
        List of results
    """
    results = []
    
    print(f"\nBatch testing {len(problems)} Game24 problems...\n")
    
    for i, problem in enumerate(problems, 1):
        print(f"Problem {i}/{len(problems)}: {problem}")
        
        # Create test messages
        test_msgs = few_shot_messages.copy()
        test_msgs.append({
            'role': 'user',
            'content': f"Solve this Game24 problem: {problem}"
        })
        
        try:
            # Get response
            response = pipeline_message(test_msgs, client)
            parsed = parse_game24_response(response)
            
            results.append({
                'problem': problem,
                'strategy': parsed['strategy'],
                'answer': parsed['answer'],
                'full_response': response
            })
            
            print(f"  ✓ Strategy: {parsed['strategy'][:50]}...")
            print(f"  ✓ Answer: {parsed['answer'][:80]}...\n")
        
        except Exception as e:
            print(f"  ✗ Error: {e}\n")
            continue
    
    print(f"✓ Completed {len(results)} problems")
    return results


# Define test problems
test_problems = [
    "1 2 3 4",      # Easy
    "2 3 5 6",      # Medium
    "3 3 8 8",      # Medium
    "1 5 5 5",      # Hard
    "4 4 6 7"       # Hard
]

# Run batch test
if len(few_shot_messages) > 1:  # Only if we have few-shot examples
    batch_results = batch_test_game24(client, test_problems[:3], few_shot_messages)  # Test first 3 to save time
else:
    print("⚠ Skipping batch test - no few-shot examples available")

## 10. Strategy Analysis

Analyze which strategies were used across different problems.

In [ ]:
if 'batch_results' in locals() and len(batch_results) > 0:
    print("\n" + "=" * 70)
    print("STRATEGY USAGE ANALYSIS:")
    print("=" * 70)
    
    for result in batch_results:
        print(f"\nProblem: {result['problem']}")
        print(f"  Strategy: {result['strategy']}")
        print(f"  Solution: {result['answer'][:100]}..." if len(result['answer']) > 100 else f"  Solution: {result['answer']}")
    
    print("\n" + "=" * 70)
    
    # Count strategy types
    strategies = [r['strategy'] for r in batch_results]
    known = sum(1 for s in strategies if any(k in s for k in ['CoT', 'ToT', 'AoT']))
    novel = len(strategies) - known
    
    print(f"\nStrategy Distribution:")
    print(f"  Known strategies (CoT/ToT/AoT): {known}")
    print(f"  Novel strategies: {novel}")
    print("=" * 70)
else:
    print("\n⚠ No batch results to analyze")

## Summary: Key Findings from Game24 Experiments

### What We Demonstrated:

1. **Basic Solving**: Model can solve standard 4-number Game24 problems

2. **Strategy Selection**: Meta-reasoning allows the model to choose appropriate strategies

3. **Trial Tracking**: The structured format helps track attempts and learn from failures

4. **Strategy Invention**: When problems become harder (5+ numbers), model can invent new approaches

5. **Iterative Refinement**: Model can verify and correct its own solutions

6. **Few-Shot Learning**: Examples from the dataset improve solution quality

### Why Game24 is a Good Benchmark:

- **Clear success criteria**: Either you get 24 or you don't
- **Multiple solutions**: Encourages exploration
- **Variable difficulty**: Can test from easy to very hard
- **Requires planning**: Can't just try random operations
- **Benefits from backtracking**: Dead ends are common

### Performance Patterns:

- **Easy problems** (e.g., 1 2 3 4): Usually solved with simple CoT
- **Medium problems** (e.g., 3 3 8 8): May use ToT for exploration
- **Hard problems** (e.g., 5 numbers): Often triggers strategy invention

### Next Steps:

- Quantitative evaluation on larger Game24 datasets
- Compare success rates: meta-reasoning vs fixed strategies
- Analyze novel strategies that emerge
- Test on even harder variations (6+ numbers)

### Related Files:

- `01_meta_learning_demo.ipynb`: General meta-learning approach
- `03_testing_reasoning.ipynb`: Testing across multiple domains
- `meta_learn.py`: Production meta-learning script
- `main.py`: Batch data generation